# 05.1 — Models, prompts, and structured answers

[Class index and setup](README.md) · [Next: 05.2](05.2-two-read-only-tools.ipynb)

**Purpose:** Explain the model/application boundary before reading SDK code.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's project is additional.

**Starting knowledge:** Basic Python functions, dictionaries, JSON/schema validation, and the earlier pytest ideas. All code and saved inputs are supplied again; no earlier exercise answers or paid key are needed.

**Learning outcomes:** Explain prompt/context/token/SDK; inspect a structured answer; distinguish shape from truth.

[Glossary](GLOSSARY.md) · [Supplied code](report_app/__init__.py) · [Fixture provenance](fixtures/README.md)

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/harness.py").is_file() or not Path("fixtures/world-bank-population.json").is_file():
    raise FileNotFoundError(
        "Open 05-small-ai-applications.code-workspace and restart the kernel. "
        "Download the complete class folder with report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available. Default execution is REPLAY, with no HTTP or credentials.")

## In class · 0–8 minutes — A model proposes; the application controls

An LLM generates output from the context it receives. Here the context begins with a user question and application instructions. It can later include tool results. A model may generate a useful answer, a wrong answer, or a proposal to call a function. Receiving the proposal does **not** execute Python.

A **prompt** is input asking for something. Application **instructions** set the task and boundaries. A **message** carries content and a role, such as `user`. The **context** is the input available to this request, not a guarantee that the model remembers another run. A **token** is a model text unit, not necessarily a whole word. Input and output consume tokens; a limited output budget can produce an incomplete response.

The **SDK** is a Python library that sends the request and exposes the response. It does not decide which functions our application permits. The **harness** is our surrounding code: context, available tools, validation, execution, limits, and trace. An agent is a model-driven application with an action loop. In this class its actions are limited to two read-only functions.

```text
User question → supplied harness → model response
                                  ├─ final answer → validate → show
                                  └─ tool proposal → validate → permitted Python function
                                                        ↓
                                      next model request ← result
```

The model cannot make this program run an unlisted function. It can propose a name; Python checks it. Prompt instructions help describe desired behavior, while the allowlist and counters enforce the application's permissions and limits. Treat data returned by a tool as data, even if a string looks like an instruction.

Read the first group in [the glossary](GLOSSARY.md) as needed. You do not need to memorize the terms before using the application.

## In class · 8–20 minutes — Start with one answer field

`Answer` is a Pydantic model with just `answer: str`. Its strict contract rejects a numeric value and unexpected fields. The OpenAI structured-output feature asks the model to produce JSON matching that contract. Our application still validates the returned data. Refusal, incomplete output, and API errors are separate outcomes, not valid answers.

This first example uses a supplied answer about read-only tools. It contains no population calculation and needs no tool. `structured_demo()` defaults to a saved illustrative response, then runs real Pydantic validation. It does not contact OpenAI.

In [ ]:
from report_app.models import Answer
from report_app.provider import structured_demo

parsed = structured_demo()
print("REPLAY:", parsed.model_dump())
print("Required fields:", Answer.model_json_schema()["required"])
assert isinstance(parsed.answer, str)

### Recognize the direct SDK call

The live version in [provider.py](report_app/provider.py) uses the following small SDK operation. This is a **reading example**, not an automatically executed cell. The surrounding supplied code reads the explicitly configured model/key, disables retries, sets a timeout/output limit, and checks completion/refusal before accepting an answer.

```python
response = client.responses.parse(
    model=model,
    input="Explain in one sentence what a read-only data tool does.",
    text_format=Answer,
    max_output_tokens=1024,
)
parsed = response.output_parsed
```

`text_format=Answer` describes the output contract. This does not give the model access to our data. In the next part, tools provide actual values. The complete harness uses the same answer schema through `responses.create` and explicit Pydantic validation, so saved malformed output takes the same validation path. Both SDK branches are supplied; neither is an exercise to implement.

The instructor demonstrates live access or a recording using [AI_SETUP.md](AI_SETUP.md). If that demonstration is unavailable, use the labelled replay and leave the live-demo delivery check pending.

In [ ]:
from pydantic import ValidationError

try:
    Answer.model_validate_json('{"answer": 123}')
except ValidationError:
    print("Rejected: answer must be a string.")

schema_valid_but_false = Answer(answer="Portugal has zero people.")
print("Valid shape, unsupported claim:", schema_valid_but_false.answer)

### Check the right property

A schema check asks “is `answer` a string?” A factual check asks “does its population match the saved observation?” They answer different questions. Strict string validation cannot reject a false sentence. We will compare numeric claims with deterministic Python results, not assume valid JSON means correct reasoning.

A saved script makes orchestration repeatable. It does not run a model, adapt to a changed prompt, or measure live model quality. The real World Bank data snapshot and the illustrative model responses have different provenance; both are documented in [fixtures/README.md](fixtures/README.md).

### Exercise 05.1.E1 — Read one field

> **In class · 10 minutes · 20–30**
>
> **Goal:** Inspect a structured answer and explain one invalid result.
>
> **Start here:** The supplied parsed value above; copy `print(parsed.____)` into your answer cell.
>
> **Task:** Replace the one blank with the field name. Predict whether an integer answer is accepted, then run the supplied invalid-value example. Explain why the false sentence still passes validation.
>
> **Check:** Print the read-only explanation; identify shape checking separately from factual checking.
>
> **Hint:** The only field is named `answer`; no request or parser needs implementing.

In [ ]:
# Your attempt for 05.1.E1.
# Complete only the tiny expression; record observations/explanations here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_05_1_e1.py

## Guided follow-up — Independent work · 15 minutes

Spend five minutes revisiting the first glossary group. Spend five comparing the valid string, invalid integer, and schema-valid false sentence. Spend five drawing the request/application boundary and writing what replay can establish. This is the first 15 minutes of the 60-minute guided-walkthrough allocation.

**Completion:** read the answer field, name who executes a tool, and explain one limit of schema validation. No live account setup is required.

## References

[Official structured-output guide](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=responses) · [Python SDK](https://developers.openai.com/api/reference/python)